In [1]:
# ─────────────────────────────────────────────────────────────
# DIAGNOSTIC — waiting_periods anchor sweep
# Source: LSDB Basis → "WP" anchor
# Schema target: product | wp_months | inc_adj
# ─────────────────────────────────────────────────────────────
from pathlib import Path
from openpyxl import load_workbook

BASE   = Path(r"J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine")
SOURCE = BASE / "Basis and Assumptions.xlsx"

wb = load_workbook(SOURCE, data_only=True, read_only=False)
ws = wb["LSDB Basis"]

def find_anchors(ws, text):
    hits = []
    target = text.strip()
    for row in ws.iter_rows():
        for cell in row:
            v = cell.value
            if isinstance(v, str) and v.strip() == target:
                hits.append((cell.row, cell.column))
    return hits

def dump_block(ws, r0, c0, n_rows=12, n_cols=4, label=""):
    print(f"\n── {label}  anchor at (row={r0}, col={c0})  sheet='{ws.title}' ──")
    print(f"{'row':>4} | " + " | ".join(f"c{c0+i:>2}" for i in range(n_cols)))
    print("-" * (7 + n_cols * 12))
    for dr in range(n_rows):
        r = r0 + dr
        vals = []
        for dc in range(n_cols):
            v = ws.cell(row=r, column=c0 + dc).value
            if v is None:
                vals.append("·")
            elif isinstance(v, float):
                vals.append(f"{v:.4g}")
            else:
                vals.append(str(v)[:10])
        print(f"{r:>4} | " + " | ".join(f"{s:>10}" for s in vals))

# Probe multiple candidate anchors
for label in ["WP", "Waiting Period", "Waiting periods", "Waiting", "WP Loading", "WP Factor"]:
    hits = find_anchors(ws, label)
    if hits:
        print(f"\nLSDB Basis — {label!r} hits: {hits}")
        for i, (r, c) in enumerate(hits, 1):
            dump_block(ws, r, c - 1, n_rows=10, n_cols=4, label=f"{label} #{i}")
    else:
        print(f"LSDB Basis — {label!r}: no hits")


LSDB Basis — 'WP' hits: [(2, 5)]

── WP #1  anchor at (row=2, col=4)  sheet='LSDB Basis' ──
 row | c 4 | c 5 | c 6 | c 7
-------------------------------------------------------
   2 |          · |         WP |    Inc Adj |          ·
   3 |          · |          0 |       1.05 |          ·
   4 |          · |          6 |          1 |          ·
   5 |          · |         12 |       0.95 |          ·
   6 |          · |         18 |        0.9 |          ·
   7 |          · |         24 |       0.85 |          ·
   8 |          · |          · |          · |          ·
   9 |          · |          · |          · |          ·
  10 |          · |          · |          · |          ·
  11 |          · |          · |          · |          ·
LSDB Basis — 'Waiting Period': no hits
LSDB Basis — 'Waiting periods': no hits
LSDB Basis — 'Waiting': no hits
LSDB Basis — 'WP Loading': no hits
LSDB Basis — 'WP Factor': no hits


In [2]:
# ═══════════════════════════════════════════════════════════════
# build_waiting_periods.ipynb — Cell 1: Setup
# ═══════════════════════════════════════════════════════════════
# Builds the 'waiting_periods' sheet of the Basis Master.
# Schema: product | wp_months | inc_adj
# Products: lsdb only (future-proof for others)
# Expected rows: 5
# ═══════════════════════════════════════════════════════════════

import csv
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import polars as pl
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

# ── Paths ──────────────────────────────────────────────────────
BASE          = Path(r"J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine")
SOURCE        = BASE / "Basis and Assumptions.xlsx"
MASTER_XLSX   = BASE / "basis_master" / "basis_master_v2026_07.xlsx"
SNAPSHOT_DIR  = BASE / "basis_snapshots" / "basis_v2026_07"
BUILD_LOG     = SNAPSHOT_DIR / "build_log.csv"

# ── Sheet identity ─────────────────────────────────────────────
SHEET_NAME    = "waiting_periods"
PARQUET_PATH  = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"
BASIS_VERSION = "B2026_07"
BUILT_BY      = "Mohamed Hussein Gadiwala"

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_XLSX.parent.mkdir(parents=True, exist_ok=True)

assert SOURCE.exists(),       f"Source workbook not found: {SOURCE}"
assert SNAPSHOT_DIR.exists(), f"Snapshot dir missing:      {SNAPSHOT_DIR}"

if MASTER_XLSX.exists():
    try:
        with open(MASTER_XLSX, "a"):
            pass
    except PermissionError:
        raise PermissionError(f"'{MASTER_XLSX.name}' is open in Excel. Close it and re-run.")

print("✅ Cell 1 — Setup complete")
print(f"   Target sheet  : {SHEET_NAME}")

✅ Cell 1 — Setup complete
   Target sheet  : waiting_periods


In [3]:
# ═══════════════════════════════════════════════════════════════
# build_waiting_periods.ipynb — Cell 2: Helpers
# ═══════════════════════════════════════════════════════════════

def find_anchors(ws, text):
    hits = []
    target = text.strip()
    for row in ws.iter_rows():
        for cell in row:
            v = cell.value
            if isinstance(v, str) and v.strip() == target:
                hits.append((cell.row, cell.column))
    return hits


def as_str(v):
    if v is None:
        return None
    s = str(v).strip()
    return s if s else None


def as_float(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).strip()
    if not s:
        return None
    try:
        return float(s)
    except ValueError:
        return None


def as_int(v):
    if v is None:
        return None
    if isinstance(v, int) and not isinstance(v, bool):
        return v
    if isinstance(v, float):
        return int(v) if v.is_integer() else None
    s = str(v).strip()
    if not s:
        return None
    try:
        return int(s)
    except ValueError:
        try:
            f = float(s)
            return int(f) if f.is_integer() else None
        except ValueError:
            return None


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


def utc_now_iso():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


print("✅ Cell 2 — Helpers loaded")

✅ Cell 2 — Helpers loaded


In [4]:
# ═══════════════════════════════════════════════════════════════
# build_waiting_periods.ipynb — Cell 3: SPECS
# ═══════════════════════════════════════════════════════════════
# Anchor "WP" at (2, 5) on LSDB Basis.
#   col 5 = WP header       → wp_months values below
#   col 6 = Inc Adj header  → inc_adj values below
# Walk down col 5 until blank.
# ═══════════════════════════════════════════════════════════════

SPECS = [
    {
        "product":       "lsdb",
        "sheet":         "LSDB Basis",
        "anchor_text":   "WP",
        "anchor_occ":    1,
        "wp_offset":     0,      # same col as anchor
        "adj_offset":    1,      # one col right
        "expected_rows": 5,
    },
]

wb = load_workbook(SOURCE, data_only=True, read_only=False)

print("🔍 Anchor validation:")
for spec in SPECS:
    ws = wb[spec["sheet"]]
    hits = find_anchors(ws, spec["anchor_text"])
    occ = spec["anchor_occ"]
    assert len(hits) >= occ, (
        f"[{spec['product']}] expected ≥{occ} '{spec['anchor_text']}' "
        f"hits on '{spec['sheet']}', got {len(hits)}: {hits}"
    )
    r, c = hits[occ - 1]
    print(f"   {spec['product']:<6} → {spec['sheet']:<12} "
          f"anchor at (row={r}, col={c})  "
          f"wp@col{c + spec['wp_offset']}  adj@col{c + spec['adj_offset']}")

print("\n✅ Cell 3 — SPECS loaded, anchor resolved")

🔍 Anchor validation:
   lsdb   → LSDB Basis   anchor at (row=2, col=5)  wp@col5  adj@col6

✅ Cell 3 — SPECS loaded, anchor resolved


In [5]:
# ═══════════════════════════════════════════════════════════════
# build_waiting_periods.ipynb — Cell 4: Extract
# ═══════════════════════════════════════════════════════════════

all_rows = []
MAX_WALK = 30

for spec in SPECS:
    ws = wb[spec["sheet"]]
    hits = find_anchors(ws, spec["anchor_text"])
    anchor_row, anchor_col = hits[spec["anchor_occ"] - 1]

    wp_col  = anchor_col + spec["wp_offset"]
    adj_col = anchor_col + spec["adj_offset"]

    r = anchor_row + 1
    rows_extracted = 0
    while r < anchor_row + 1 + MAX_WALK:
        wp = as_int(ws.cell(row=r, column=wp_col).value)
        if wp is None:
            break
        adj = as_float(ws.cell(row=r, column=adj_col).value)
        all_rows.append({
            "product":   spec["product"],
            "wp_months": wp,
            "inc_adj":   adj,
        })
        rows_extracted += 1
        r += 1

    print(f"   {spec['product']:<6} → {rows_extracted} rows extracted "
          f"(expected {spec['expected_rows']})")

df = pl.DataFrame(all_rows, infer_schema_length=None)

print(f"\n✅ Cell 4 — Extract complete")
print(f"   Total rows : {df.height}")
print(f"   Schema     : {df.schema}")
print("\n── Full table ──")
print(df)

   lsdb   → 5 rows extracted (expected 5)

✅ Cell 4 — Extract complete
   Total rows : 5
   Schema     : Schema([('product', String), ('wp_months', Int64), ('inc_adj', Float64)])

── Full table ──
shape: (5, 3)
┌─────────┬───────────┬─────────┐
│ product ┆ wp_months ┆ inc_adj │
│ ---     ┆ ---       ┆ ---     │
│ str     ┆ i64       ┆ f64     │
╞═════════╪═══════════╪═════════╡
│ lsdb    ┆ 0         ┆ 1.05    │
│ lsdb    ┆ 6         ┆ 1.0     │
│ lsdb    ┆ 12        ┆ 0.95    │
│ lsdb    ┆ 18        ┆ 0.9     │
│ lsdb    ┆ 24        ┆ 0.85    │
└─────────┴───────────┴─────────┘


In [6]:
# ═══════════════════════════════════════════════════════════════
# build_waiting_periods.ipynb — Cell 5: Sanity checks
# ═══════════════════════════════════════════════════════════════

# Structural
assert df.height == 5, f"expected 5 rows, got {df.height}"
assert set(df.columns) == {"product", "wp_months", "inc_adj"}
assert df["product"].n_unique() == 1
assert df["wp_months"].null_count() == 0
assert df["inc_adj"].null_count() == 0

# WP durations
expected_wps = [0, 6, 12, 18, 24]
actual_wps = sorted(df["wp_months"].to_list())
assert actual_wps == expected_wps, f"wp_months: {actual_wps}"
print(f"   ✅ wp_months values : {actual_wps}")

# Value spot-checks
SPOT_CHECKS = [
    ("lsdb",  0, 1.05),
    ("lsdb",  6, 1.00),
    ("lsdb", 12, 0.95),
    ("lsdb", 18, 0.90),
    ("lsdb", 24, 0.85),
]

TOL = 1e-6
print("\n🎯 Spot checks:")
fails = 0
for product, wp, expected in SPOT_CHECKS:
    got = df.filter(
        (pl.col("product") == product) & (pl.col("wp_months") == wp)
    )["inc_adj"][0]
    ok = got is not None and abs(got - expected) < TOL
    flag = "✅" if ok else "❌"
    print(f"   {flag}  {product} wp={wp:>2} → expected={expected:<5} got={got}")
    if not ok:
        fails += 1
assert fails == 0, f"{fails} spot check(s) failed"

# Advisory: check linear pattern (–0.05 per 6 months)
sorted_df = df.sort("wp_months")
diffs = [
    round(sorted_df["inc_adj"][i+1] - sorted_df["inc_adj"][i], 4)
    for i in range(sorted_df.height - 1)
]
print(f"\n📌 Note: inc_adj deltas between successive WP durations = {diffs}")
print(f"   (linear –0.05 per 6-month step)")

print("\n✅ Cell 5 — All sanity checks passed")

   ✅ wp_months values : [0, 6, 12, 18, 24]

🎯 Spot checks:
   ✅  lsdb wp= 0 → expected=1.05  got=1.05
   ✅  lsdb wp= 6 → expected=1.0   got=1.0
   ✅  lsdb wp=12 → expected=0.95  got=0.95
   ✅  lsdb wp=18 → expected=0.9   got=0.9
   ✅  lsdb wp=24 → expected=0.85  got=0.85

📌 Note: inc_adj deltas between successive WP durations = [-0.05, -0.05, -0.05, -0.05]
   (linear –0.05 per 6-month step)

✅ Cell 5 — All sanity checks passed


In [7]:
# ═══════════════════════════════════════════════════════════════
# build_waiting_periods.ipynb — Cell 6: Write outputs + build log
# ═══════════════════════════════════════════════════════════════

# Parquet
df.write_parquet(PARQUET_PATH)
parquet_hash = sha256_of(PARQUET_PATH)
parquet_kb   = round(PARQUET_PATH.stat().st_size / 1024, 2)
print(f"📦 Parquet written : {PARQUET_PATH.name}  ({parquet_kb} KB)")

# Master xlsx guard
if MASTER_XLSX.exists():
    try:
        with open(MASTER_XLSX, "a"):
            pass
    except PermissionError:
        raise PermissionError(f"'{MASTER_XLSX.name}' is open in Excel. Close it and re-run.")

if MASTER_XLSX.exists():
    wb_master = load_workbook(MASTER_XLSX)
else:
    from openpyxl import Workbook
    wb_master = Workbook()
    if "Sheet" in wb_master.sheetnames and len(wb_master.sheetnames) == 1:
        del wb_master["Sheet"]

if SHEET_NAME in wb_master.sheetnames:
    del wb_master[SHEET_NAME]

ws_new = wb_master.create_sheet(SHEET_NAME)
columns = df.columns
ws_new.append(columns)
for row in df.iter_rows():
    ws_new.append(list(row))

header_row = next(ws_new.iter_rows(min_row=1, max_row=1))
for cell in header_row:
    cell.font      = Font(bold=True, color="FFFFFF")
    cell.fill      = PatternFill("solid", fgColor="305496")
    cell.alignment = Alignment(horizontal="left")

for col_idx, col_name in enumerate(columns, start=1):
    max_len = max(
        [len(str(col_name))] +
        [len(str(v)) if v is not None else 0 for v in df[col_name].to_list()]
    )
    ws_new.column_dimensions[get_column_letter(col_idx)].width = max(12, max_len + 2)

last_col_letter = get_column_letter(len(columns))
table_ref = f"A1:{last_col_letter}{df.height + 1}"
tbl = Table(displayName=f"tbl_{SHEET_NAME}", ref=table_ref)
tbl.tableStyleInfo = TableStyleInfo(
    name="TableStyleMedium2",
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=False,
)
ws_new.add_table(tbl)

wb_master.save(MASTER_XLSX)
print(f"📗 Master xlsx     : sheet '{SHEET_NAME}' upserted")

# Build log
source_hash = sha256_of(SOURCE)

log_row = {
    "timestamp_utc":     utc_now_iso(),
    "sheet":             SHEET_NAME,
    "row_count":         df.height,
    "distinct_products": df["product"].n_unique(),
    "source_file":       SOURCE.name,
    "source_sha256":     source_hash,
    "parquet_file":      PARQUET_PATH.name,
    "parquet_sha256":    parquet_hash,
    "parquet_kb":        parquet_kb,
    "status":            "ok",
    "built_by":          BUILT_BY,
}

log_exists = BUILD_LOG.exists()
with open(BUILD_LOG, "a", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=list(log_row.keys()))
    if not log_exists:
        writer.writeheader()
    writer.writerow(log_row)

print(f"📝 Build log       : appended row for '{SHEET_NAME}'")
print(f"\n✅ Cell 6 — waiting_periods build COMPLETE")
print(f"   Rows : {df.height}, Product : lsdb")

📦 Parquet written : waiting_periods.parquet  (1.18 KB)
📗 Master xlsx     : sheet 'waiting_periods' upserted
📝 Build log       : appended row for 'waiting_periods'

✅ Cell 6 — waiting_periods build COMPLETE
   Rows : 5, Product : lsdb
